# Where embeddings live

**What you'll learn.** embpy always writes embeddings into a predictable,
AnnData-native place, records how they were made, and leaves your `.X` alone.
Once you know the contract, every embedding is easy to find, trust, and reload.

Three rules:

- **row-aligned** embeddings → `.obsm` (one vector per observation)
- **feature-aligned** embeddings → `.varm` (one vector per variable)
- **payload + provenance** → `.uns` (the entity-level vectors and how they were made)

`.X` is reserved for your counts / expression and is never overwritten.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd

from embpy import BioEmbedder

embedder = BioEmbedder(device="auto", organism="human")
genes = ["TP53", "EGFR", "MYC", "BRCA1", "JUN", "STAT1"]

## Row-aligned embeddings → `.obsm`

When each **row** of your AnnData is the thing you're embedding — a gene, a
cell, a sample — the embedding is row-aligned and lands in `.obsm`, exactly
like a PCA or UMAP basis. It has one vector per `obs`, matched to `obs_names`.

In [ ]:
rows = ad.AnnData(
    X=np.zeros((len(genes), 1), dtype=np.float32),
    obs=pd.DataFrame({"symbol": genes}, index=genes),
)
rows = embedder.embed(
    rows, entity_type="gene", id_type="symbol", obs_column="symbol",
    model="genept", output="anndata", key="X_rows",
)

print("obsm keys :", list(rows.obsm.keys()))
print("shape     :", rows.obsm["X_rows"].shape, "→ one row per observation")

## Feature-aligned embeddings → `.varm`

When you embed the **features** (genes as columns of an expression matrix), the
embedding is feature-aligned and lands in `.varm` — one vector per `var`, so it
stays matched to your gene axis.

In [ ]:
feat = ad.AnnData(
    X=np.zeros((1, len(genes)), dtype=np.float32),
    obs=pd.DataFrame(index=["example_cell"]),
    var=pd.DataFrame({"gene_symbol": genes}, index=pd.Index(genes, name="gene_symbol")),
)
feat = embedder.embed(
    feat, entity_type="gene", id_type="symbol", var_column="gene_symbol",
    model="genept", output="anndata", key="X_gene_feature",
)

print("varm keys :", list(feat.varm.keys()))
print("shape     :", feat.varm["X_gene_feature"].shape, "→ one row per gene (var)")

## Payload + provenance → `.uns`

embpy records what it did in `.uns` so an embedding is reproducible and
self-describing: which model produced it, the pooling, the resolved identifiers.
You never have to guess where a `.obsm` matrix came from.

In [ ]:
meta = rows.uns.get("embeddings", rows.uns)
print("provenance recorded under .uns:")
for k in list(meta.keys())[:8]:
    print("  ", k)

## Save once, reload anywhere

Because everything lives in the AnnData, a single `write_h5ad` persists your
embeddings *and* their provenance. Reload and they're exactly where you left
them — no recomputation, no lost metadata.

In [ ]:
from pathlib import Path

out = Path("outputs"); out.mkdir(exist_ok=True)
rows.write_h5ad(out / "gene_embeddings.h5ad")

reloaded = ad.read_h5ad(out / "gene_embeddings.h5ad")
print("embeddings survived the round-trip:", "X_rows" in reloaded.obsm)
print("shape preserved:", reloaded.obsm["X_rows"].shape)

## Summary

- Row-aligned → `.obsm`, feature-aligned → `.varm`, payload/provenance → `.uns`.
- `.X` is always your data, never an embedding.
- One `write_h5ad` saves the vectors and how they were made.

**Next:** [Compare embedding spaces across models](03_compare_models.ipynb).